# Loan Approval Classifier (HMDA mortgage applications)

Supervised classification on real mortgage decisions. The data is the 2022 public HMDA (Home Mortgage Disclosure Act) loan-level file for Delaware, pulled from the CFPB HMDA Data Browser API. US lenders have to report every mortgage application and its outcome, so this is about as real as public loan-approval data gets. I keep the two clear outcomes: originated (approved, 31,543 rows) and denied (10,032 rows), about 41.6K applications total.

Features are what a lender sees at application time: loan amount, income, loan term, property value, loan-to-value, debt-to-income bucket, loan type and purpose, lien status, occupancy, applicant age band, and which credit score model was used. I leave out anything that only exists after a decision (interest rate, fees, purchaser, denial reasons) because those would leak the answer. Race, ethnicity and sex are not model inputs; I only use them afterwards to look at approval rates.

Two models: logistic regression (the right starting point for tabular data) and a random forest for comparison.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

np.random.seed(1)
CSV = Path("../../data/loan_approval/hmda_de_2022.csv")
# CFPB HMDA Data Browser API: Delaware, 2022, actions 1 (originated) and 3 (denied)
URL = "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=DE&years=2022&actions_taken=1,3"
if not CSV.exists():
    CSV.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, CSV)
raw = pd.read_csv(CSV, low_memory=False)

num_cols = ["loan_amount", "income", "loan_term", "property_value", "loan_to_value_ratio",
            "tract_to_msa_income_percentage"]
cat_cols = ["debt_to_income_ratio", "loan_type", "loan_purpose", "lien_status", "occupancy_type",
            "applicant_age", "applicant_credit_score_type"]

df = pd.DataFrame({"approved": (raw["action_taken"] == 1).astype(int)})
for c in num_cols:
    v = pd.to_numeric(raw[c], errors="coerce")          # "Exempt" and blanks become NaN
    df[c + "_missing"] = v.isna().astype(int)
    df[c] = v.fillna(v.median())
df["log_income"] = np.log1p(df["income"].clip(lower=0))
df["log_loan_amount"] = np.log1p(df["loan_amount"])
for c in cat_cols:
    df = df.join(pd.get_dummies(raw[c].astype(str).fillna("NA"), prefix=c, drop_first=True).astype(int))
print(raw.shape, "->", df.shape, " approval rate:", df.approved.mean().round(3))
df.head()

(41575, 99) -> (41575, 63)  approval rate: 0.759


,approved,loan_amount_missing,loan_amount,income_missing,income,loan_term_missing,loan_term,property_value_missing,property_value,loan_to_value_ratio_missing,...,applicant_credit_score_type_11,applicant_credit_score_type_1111,applicant_credit_score_type_2,applicant_credit_score_type_3,applicant_credit_score_type_4,applicant_credit_score_type_5,applicant_credit_score_type_6,applicant_credit_score_type_7,applicant_credit_score_type_8,applicant_credit_score_type_9
0,1,0,365000.0,0,72.0,0,360.0,0,385000.0,0,...,0,0,0,1,0,0,0,0,0,0
1,1,0,305000.0,0,107.0,0,360.0,0,315000.0,0,...,0,0,0,1,0,0,0,0,0,0
2,1,0,545000.0,0,279.0,0,360.0,0,725000.0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,505000.0,0,206.0,0,360.0,0,585000.0,0,...,0,0,1,0,0,0,0,0,0,0
4,1,0,195000.0,0,47.0,0,360.0,0,385000.0,0,...,0,0,1,0,0,0,0,0,0,0


In [2]:
X = df.drop(columns=["approved"]).values.astype(float)
y = df["approved"].values
features = df.drop(columns=["approved"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)
sc = StandardScaler().fit(X_train)
X_train_s, X_test_s = sc.transform(X_train), sc.transform(X_test)

lr = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=1, n_jobs=2).fit(X_train, y_train)

for name, model, Xte in [("LogReg", lr, X_test_s), ("RandomForest", rf, X_test)]:
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"--- {name} ---  AUC={roc_auc_score(y_test, proba):.3f}")
    print(classification_report(y_test, pred, target_names=["denied", "approved"], digits=3))

--- LogReg ---  AUC=0.835
              precision    recall  f1-score   support

      denied      0.811     0.447     0.576      2006
    approved      0.846     0.967     0.902      6309

    accuracy                          0.841      8315
   macro avg      0.828     0.707     0.739      8315
weighted avg      0.838     0.841     0.824      8315

--- RandomForest ---  AUC=0.876
              precision    recall  f1-score   support

      denied      0.892     0.488     0.630      2006
    approved      0.858     0.981     0.915      6309

    accuracy                          0.862      8315
   macro avg      0.875     0.734     0.773      8315
weighted avg      0.866     0.862     0.846      8315



In [3]:
# coefficient signs from logreg should make basic sense; show the 8 most negative and 8 most positive
coef = pd.Series(lr.coef_[0], index=features).sort_values()
print(pd.concat([coef.head(8), coef.tail(8)]).round(3))

# descriptive look only: raw approval rate by reported race and sex (not model inputs)
for col in ["derived_race", "derived_sex"]:
    print()
    print(raw.groupby(col)["action_taken"].agg(n="size", approval_rate=lambda s: (s == 1).mean()).round(3)
             .sort_values("n", ascending=False).head(6))

debt_to_income_ratio_>60%          -1.069
loan_purpose_32                    -0.528
loan_to_value_ratio                -0.429
loan_purpose_4                     -0.418
loan_purpose_2                     -0.396
debt_to_income_ratio_50%-60%       -0.340
applicant_credit_score_type_8      -0.290
loan_to_value_ratio_missing        -0.281
income_missing                      0.122
log_income                          0.144
applicant_age_8888                  0.150
applicant_credit_score_type_7       0.157
applicant_credit_score_type_1111    0.173
debt_to_income_ratio_Exempt         0.173
lien_status_2                       0.255
log_loan_amount                     0.383
dtype: float64

                                      n  approval_rate
derived_race                                          
White                             24673          0.798
Black or African American          7181          0.660
Race Not Available                 6856          0.720
Asian                              17

                       n  approval_rate
derived_sex                            
Joint              14151          0.834
Male               12944          0.727
Female             11124          0.721
Sex Not Available   3356          0.689


## Results and a note on fairness

On the 8,315 held-out applications, the random forest reaches ROC-AUC 0.876 and 86.2% accuracy, ahead of logistic regression (AUC 0.835, 84.1%). Both models are much better at confirming approvals than at catching denials: the forest only flags 49% of the denied applications at the default 0.5 threshold, so a lender using this for triage would want a lower threshold or class weights.

The logistic coefficients make basic sense. A debt-to-income ratio above 60% is the strongest push toward denial, followed by cash-out refinancing (HMDA loan purpose 32), a higher loan-to-value ratio, and home-improvement or "other" purpose loans compared to home purchase. One caveat: the missing-value flags carry some signal (for example a missing loan-to-value ratio leans toward denial). These fields are blank far more often for denied applications, probably because some never got far enough for them to be filled in, so part of that signal is likely a side effect of how the form gets filled in, not something a lender could use at application time.

Loan approval is one of those domains where the *modeling* part is the easy part. The descriptive table above shows why the rest matters: in this Delaware 2022 data, raw approval rates are 79.8% for White applicants and 66.0% for Black applicants. Those raw gaps don't control for income, debt-to-income or loan type, so they aren't proof of discrimination on their own, but any model trained on these historical decisions will learn whatever is behind them.

For an actual fintech project I'd at minimum:
- Run group-fairness checks (demographic parity, equalized odds) across protected attributes, using HMDA's reported race, ethnicity and sex.
- Avoid using census tract or other proxies for race. I kept `tract_to_msa_income_percentage` here for a first pass, but it's a geographic feature and would need a closer look.
- Use a model with built-in explanations (SHAP), since applicants in the US have a right to the principal reasons for an adverse decision.